# 태스크 3: Amazon Bedrock AgentCore에 에이전트 배포

이 태스크에서는 Amazon Bedrock AgentCore를 사용하여 로컬 개발에서 프로덕션 배포로 전환합니다. 태스크 1과 태스크 2에 구축된 정교한 다중 에이전트 재무 자문 시스템을 AWS의 엔터프라이즈급 에이전트 호스팅 플랫폼인 AgentCore Runtime에 배포하게 됩니다.

![아키텍처](./images/architecture_ko_kr.png)


## 학습 내용

엔터프라이즈 애플리케이션에 필요한 보안, 성능 및 안정성 표준을 유지하면서 에이전트를 대규모로 실행하기 위해 특별히 구축된 AgentCore의 인프라를 활용하는 방법을 배우게 됩니다.

##Amazon Bedrock AgentCore Runtime

[Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html)은 AI 에이전트 또는 도구를 배포하고 실행할 수 있는 안전한 서버리스 전용 호스팅 환경을 제공하여 실험에서 프로덕션급 에이전트로의 가치 창출 시간을 단축합니다.

[AgentCore Runtime의 작동 방식에 대해 자세히 알아보기](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-how-it-works.html)

## Amazon Bedrock AgentCore Observability

[AgentCore Observability](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability.html)은 프로덕션 환경에서 에이전트 성능을 추적, 디버그 및 모니터링하는 데 도움이 됩니다. Observability는 에이전트 워크플로의 각 단계를 자세히 시각화하므로 에이전트 실행 경로를 검사하고 중간 출력을 감사할 수 있으며 성능 병목 현상과 장애를 디버그할 수 있습니다.

In [ ]:
%%capture
# [환경 준비] AgentCore 배포에 필요한 라이브러리 설치.
# Task 3 는 Task 1\uff652 의 budget_agent.py\uff65financial_analysis_agent.py 를 재사용하므로 순서대로 실행해야 한다.
!pip install --force-reinstall -U -r requirements.txt --quiet --disable-pip-version-check

In [ ]:
# [임포트] AgentCore Runtime 배포 도구와 인증 유틸리티.
import uuid                                              # 세션 ID 생성용(충돌 방지를 위한 고유 식별자)
from utils import setup_cognito_user_pool, reauthenticate_user  # 실습 제공: Cognito 설정 / 토큰 재발급 헬퍼
from bedrock_agentcore_starter_toolkit import Runtime    # AgentCore Runtime 배포를 돕는 스타터 툴킷
from boto3.session import Session
from typing import Any, Optional
import urllib.parse                                      # ARN 을 URL 에 넣기 위한 인코딩용
import requests                                          # 배포된 에이전트를 HTTP 로 직접 호출할 때 사용
import json

In [ ]:
# [세션/리전] 현재 자격 증명의 기본 리전을 읽는다(하드코딩 회피).
boto_session = Session()
region = boto_session.region_name

###태스크 3.1: AgentCore Runtime에 배포할 에이전트 준비

이제 에이전트를 `AgentCore Runtime`에 배포해 보겠습니다. 이를 위해서는 다음이 필요합니다.

- `from bedrock_agentcore.runtime import BedrockAgentCoreApp`을 사용해 런타임 앱을 가져옵니다.
- `app = BedrockAgentCoreApp()`에서 코드를 사용하여 앱을 초기화합니다.
- `@app.entrypoint` 데코레이터로 호출 함수를 서식합니다.
- `app.run()`으로 AgentCore Runtime이 다음과 같이 에이전트 실행을 제어하도록 합니다.

In [ ]:
# ============================================================================
# [파일 익스포트] 이 셀을 실행하면 내용이 main.py 로 '저장'된다(코드 실행 아님).
# 이 main.py 가 곧 AgentCore Runtime 에 배포되는 진입점이다.
# Task 2 의 오케스트레이터에 (1) AgentCore 앱 래핑 (2) 메모리 조회/저장 (3) 스트리밍 을 추가한 버전.
# ============================================================================
%%writefile main.py
# Create AgentCore-compatible deployment file with streaming endpoint and memory integration

from strands import Agent, tool
from strands.models import BedrockModel
from strands.agent.conversation_manager import SummarizingConversationManager

from budget_agent import FinancialReport, budget_agent
from financial_analysis_agent import financial_analysis_agent
from bedrock_agentcore import BedrockAgentCoreApp
from bedrock_agentcore.memory import MemoryClient

from utils import get_guardrail_id
import uuid
import os
import logging
import boto3

# Get the current AWS region dynamically
region = boto3.Session().region_name

# Configure logging for error tracking and debugging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Initialize AgentCore app
# [AgentCore 앱] 배포용 앱 객체. 아래 @app.entrypoint 로 지정한 함수가 호출 진입점이 된다.
app = BedrockAgentCoreApp()

ORCHESTRATOR_PROMPT = """You are a comprehensive financial advisor orchestrator that coordinates between specialized financial agents to provide complete financial guidance. 

Your specialized agents are:
1. **budget_agent**: Handles budgeting, spending analysis, savings recommendations, and expense tracking
2. **financial_analysis_agent_tool**: Handles investment analysis, stock research, portfolio creation, and performance comparisons

Guidelines for using your agents:
- Use **budget_agent** for questions about: budgets, spending habits, expense tracking, savings goals, debt management
- Use **financial_analysis_agent_tool** for questions about: stocks, investments, portfolios, market analysis, investment recommendations
- You can use both agents together for comprehensive financial planning
- Always provide a cohesive summary that combines insights from multiple agents when applicable
- Maintain a helpful, professional tone and include appropriate disclaimers about financial advice

When a user asks a question:
1. Determine which agent(s) are most appropriate
2. Call the relevant agent(s) with focused queries
3. Synthesize the responses into a coherent, comprehensive answer
4. Provide actionable next steps when possible"""

# Add conversation management to maintain context
conversation_manager = SummarizingConversationManager(
    summary_ratio=0.3,  # Summarize 30% of messages when context reduction is needed
    preserve_recent_messages=5,  # Always keep 5 most recent messages
)

# Configure Bedrock model
bedrock_model = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    region_name=region,
    temperature=0.0,  # Deterministic responses for financial advice
    guardrail_id=get_guardrail_id(),
    guardrail_version="DRAFT",
    guardrail_trace="enabled",
)


@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    logger.info(f"budget_agent_tool called with query: {query[:100]}...")
    try:
        structured_response = budget_agent.structured_output(
            output_model=FinancialReport, prompt=query
        )
        logger.info("budget_agent_tool completed successfully")
        return structured_response
    except Exception as e:
        logger.error(f"Error in budget_agent_tool: {e}", exc_info=True)
        # Return a default structured response on error
        return FinancialReport(
            monthly_income=0.0,
            budget_categories=[],
            recommendations=[f"Error generating report: {str(e)}"],
            financial_health_score=1,
        )


@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    logger.info(f"financial_analysis_agent_tool called with query: {query[:100]}...")
    try:
        response = financial_analysis_agent(query)
        logger.info("financial_analysis_agent_tool completed successfully")
        return str(response)
    except Exception as e:
        logger.error(f"Error in financial_analysis_agent_tool: {e}", exc_info=True)
        return f"❌ Financial analysis error: {str(e)}"


# Initialize orchestrator agent at module level
logger.info("Initializing orchestrator agent...")
orchestrator_agent = Agent(
    model=bedrock_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,
)
logger.info("Orchestrator agent initialized successfully")


# [진입점] AgentCore Runtime 이 InvokeAgentRuntime 요청을 받으면 이 async 함수를 실행한다.
# payload 는 호출 측이 보낸 JSON(prompt･actor_id･session_id).
@app.entrypoint
async def invoke(payload):
    """Your AI agent function with memory integration"""
    # [입력 파싱] payload 딕셔너리에서 필요한 값을 꺼낸다.
    user_message = payload["prompt"]
    actor_id = payload.get("actor_id", "default_user")  # User identifier
    session_id = payload.get("session_id", str(uuid.uuid4()))  # Session identifier
    
    # Retrieve relevant memories before processing
    # [메모리 ID] 배포 시 env_vars 로 넣은 메모리 리소스 ID를 환경 변수에서 읽는다(없으면 None).
    memory_id = os.environ.get('AGENTCORE_MEMORY_ID')
    # [기억 저장] 응답이 끝난 뒤, 이번 대화(질문+답변)를 이벤트로 저장한다(다음 대화에서 조회됨).
    if memory_id:
        try:
            memory_client = MemoryClient()
            # Query relevant memories for context
            # [기억 조회] 이번 질문과 관련된 과거 기억을 시맨틱 검색으로 가져온다(응답 생성 '전').
            relevant_memories = memory_client.query_records(
                memory_id=memory_id,
                query=user_message,
                namespace=f"finance/user/{actor_id}/facts",
                max_results=5
            )
            
            # Add memory context to system prompt if available
            # [프롬프트 보강] 관련 기억이 있으면 시스템 프롬프트 뒤에 붙여, 에이전트가 사용자를 '기억'하게 한다.
            if relevant_memories:
                memory_context = "\n\nRelevant user information from previous conversations:\n"
                for record in relevant_memories:
                    memory_context += f"- {record.get('content', '')}\n"
                
                # Enhance agent with memory context
                enhanced_prompt = ORCHESTRATOR_PROMPT + memory_context
                orchestrator_agent.system_prompt = enhanced_prompt
        except Exception as e:
            logger.warning(f"Error retrieving memories: {e}")
    
    # Stream response
    # [스트리밍] 응답 조각을 모으면서(full_response) 동시에 호출 측에 yield 로 흘려보낸다.
    full_response = ""
    # stream_async 는 비동기 제너레이터. "data" 키가 있는 이벤트만 텍스트 청크다.
    async for event in orchestrator_agent.stream_async(user_message):
        if "data" in event:
            chunk = event["data"]
            full_response += chunk
            yield chunk
    
    # Store conversation in memory after response
    if memory_id:
        try:
            memory_client.create_event(
                memory_id=memory_id,
                actor_id=actor_id,
                session_id=session_id,
                content=f"User: {user_message}\nAssistant: {full_response}"
            )
        except Exception as e:
            logger.warning(f"Error storing memory: {e}")


if __name__ == "__main__":
    app.run()

## 배후에서는 어떤 작업이 이루어집니까?

사용시 `BedrockAgentCoreApp`에서 자동으로 수행하는 작업:

** 포트 8080에서 수신 대기하는 HTTP 서버를 생성
** 에이전트의 요구 사항을 처리하는 데 필요한 `/invocations` 엔드포인트를 구현
** 상태 확인을 위한 `/ping` 엔드포인트 구현(비동기 에이전트에 매우 중요)
** 적절한 콘텐츠 유형 및 응답 형식 처리
** AWS 표준에 따라 오류 처리 관리

### 태스크 3.2: 인증을 위한 Amazon Cognito 설정

AgentCore Runtime에는 인증이 필요합니다. Amazon Cognito를 사용하여 배포한 에이전트 서버에 액세스하기 위한 JWT 토큰을 제공할 것입니다.

In [ ]:
# [인증 준비] Amazon Cognito 사용자 풀을 만든다.
# AgentCore Runtime 은 인바운드 인증으로 IAM(SigV4) 또는 OAuth 를 쓰는데, 여기서는 Cognito(OAuth/JWT)를 쓴다.
# setup_cognito_user_pool 은 실습 헬퍼로, 사용자 풀\uff65앱 클라이언트\uff65테스트 사용자를 한 번에 생성한다.
print("Setting up Amazon Cognito user pool...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed ✓")
# .get("키", "N/A"): 키가 없어도 KeyError 없이 안전하게 출력
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

In [ ]:
# [JWT 인증 설정] AgentCore Runtime 이 요청 토큰을 검증하는 규칙.
#   allowedClients: 이 클라이언트가 발급한 토큰만 허용
#   discoveryUrl  : OIDC 디스커버리 URL(토큰 서명 키\uff65발급자 정보를 여기서 얻어 검증)
# 이 auth_config 는 뒤의 runtime.configure(authorizer_configuration=...) 에 전달된다.
auth_config = {
    "customJWTAuthorizer": {
        "allowedClients": [cognito_config["client_id"]],
        "discoveryUrl": cognito_config["discovery_url"],
    }
}

### 태스크 3.2.1: AgentCore Memory 구성

AgentCore Memory를 사용하면 에이전트가 대화 전반의 컨텍스트를 기억할 수 있습니다. 세 가지 메모리 전략을 구성하겠습니다.

1. **시맨틱 전략**: 사실 정보(수입, 지출, 목표) 추출 및 저장
2. **사용자 선호도 전략**: 사용자 선호도 파악(위험 감수, 투자 스타일)
3. **요약 전략**: 컨텍스트에 대한 대화 요약 작성

이를 통해 재무 에이전트가 이전 상호 작용을 기반으로 맞춤형 조언을 제공할 수 있습니다.

In [ ]:
# [메모리 임포트] AgentCore Memory 클라이언트와 전략 타입.
# StrategyType: 장기 메모리 추출 전략의 종류(SEMANTIC/USER_PREFERENCE/SUMMARY 등)를 상수로 제공.
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

In [ ]:
# [메모리 전략 정의] 대화에서 '무엇을 장기 기억으로 추출할지' 3가지 전략을 선언한다.
# 각 전략은 추출 종류(StrategyType) + 저장 위치(namespaces)로 구성된다.
# namespaces 의 {actorId}\uff65{sessionId} 는 런타임에 실제 사용자/세션 값으로 치환되는 자리표시자.
# (모듈 1\uff652 의 '네임스페이스 설계' — 사용자별로 기억을 격리\uff65공유하는 축)
memory_strategies = [
    {
        # SEMANTIC: 대화에서 '사실'을 추출(소득\uff65지출\uff65목표 등)
        StrategyType.SEMANTIC.value: {
            "name": "financial_facts",
            "description": "Extracts and stores user's financial facts (income, expenses, goals)",
            "namespaces": ["finance/user/{actorId}/facts"]     # 사용자별 사실 저장 경로
        }
    },
    {
        # USER_PREFERENCE: 사용자의 '선호'를 포착(투자 위험 성향\uff65예산 우선순위 등)
        StrategyType.USER_PREFERENCE.value: {
            "name": "user_preferences",
            "description": "Captures user preferences for investment risk, budget priorities",
            "namespaces": ["finance/user/{actorId}/preferences"]
        }
    },
    {
        # SUMMARY: 대화 자체를 요약. 세션 단위로 저장하므로 경로에 {sessionId} 가 들어간다.
        StrategyType.SUMMARY.value: {
            "name": "conversation_summary",
            "description": "Summarizes financial advisory conversations",
            "namespaces": ["finance/user/{actorId}/sessions/{sessionId}"]
        }
    }
]

In [ ]:
# [메모리 리소스 생성] 위 전략들을 담은 AgentCore Memory 리소스를 만든다(수 분 소요).
# 이미 같은 이름의 리소스가 있으면 재사용하고, 없을 때만 새로 만든다(중복 생성 방지).
memory_client = MemoryClient()
print("Creating AgentCore Memory for financial advisor...")
try:
    # [멱등성 확보] 기존 메모리 목록을 조회해 같은 이름이 있는지 먼저 확인한다.
    existing_memories = memory_client.list_memories()
    memory_id = None
    # [반복문] 기존 리소스를 순회하며 이름이 일치하는 것을 찾는다.
    for mem in existing_memories:
        if mem.get('name') == 'FinancialAdvisorMemory':   # .get: 키 없어도 안전
            memory_id = mem['id']
            print(f"✓ Using existing memory: {memory_id}")
            break                                          # 찾았으면 더 볼 필요 없이 중단
    # [신규 생성] 기존 것이 없을 때만 만든다.
    if not memory_id:
        # create_memory_and_wait: 생성 요청 후 '준비 완료될 때까지' 블로킹 대기하는 편의 메서드
        memory = memory_client.create_memory_and_wait(
            name="FinancialAdvisorMemory",
            strategies=memory_strategies,                  # 위에서 정의한 3가지 전략 연결
            description="Memory for personal finance advisor agent",
            event_expiry_days=90  # 원시 이벤트 보관 기간(90일). 이후 자동 만료
        )
        memory_id = memory["id"]                           # 응답 딕셔너리에서 id 파싱
        print(f"✓ Memory created: {memory_id}")
        print(f"  - Semantic strategy: Extracts financial facts")
        print(f"  - User preference strategy: Captures preferences")
        print(f"  - Summary strategy: Creates conversation summaries")
except Exception as e:
    # [점진적 성능 저하] 메모리 생성이 실패해도 실습이 멈추지 않도록 memory_id=None 으로 계속 진행.
    # 이후 단계는 memory_id 가 있으면 메모리 연동, 없으면 메모리 없이 동작한다.
    print(f"❌ Error setting up memory: {e}")
    print("Continuing without memory integration...")
    memory_id = None

### 태스크 3.3: 에이전트를 AgentCore Runtime에 배포

이 `CreateAgentRuntime` 작업은 컨테이너 이미지, 환경 변수 및 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. 또한 프로토콜 설정(HTTP, MCP) 및 권한 부여 메커니즘을 구성하여 클라이언트가 에이전트와 통신하는 방식을 제어할 수 있습니다. 

**참고:** 운영 모범 사례는 코드를 컨테이너로 패키징하고 CI/CD 파이프라인 및 IaC를 사용하여 ECR로 푸시하는 것입니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 아티팩트를 쉽게 패키징하고 AgentCore Runtime에 배포할 것입니다.

### 태스크 3.3.1: AgentCore Runtime 배포 구성

먼저 스타터 도구 키트를 사용하여 진입점, 방금 생성한 실행 역할, 요구 사항 파일을 포함하는 AgentCore Runtime 배포를 구성하겠습니다. 또한 시작 시 Amazon ECR 리포지토리를 자동으로 생성하도록 스타터 도구 키트를 구성할 예정입니다.

구성 단계에서 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

![런타임](./images/runtime_overview_ko_kr.png)

In [ ]:
# [Runtime 배포 설정] 어떤 코드를, 어떤 권한\uff65인증으로 배포할지 구성한다.
import boto3
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']   # 현재 계정 ID(실행 역할 ARN 조립에 사용)

agentcore_runtime = Runtime()
agent_name = "personal_finance_agent"

print("Configuring AgentCore Runtime...")
# 사전 생성된 실행 역할의 ARN 을 계정 ID\uff65리전으로 조립한다.
execution_role_arn = f"arn:aws:iam::{account_id}:role/AmazonBedrockAgentCoreSDKRuntime-{region}"

response = agentcore_runtime.configure(
    entrypoint="main.py",                 # 배포할 진입점 파일(위에서 %%writefile 로 만든 것)
    execution_role=execution_role_arn,    # 에이전트가 다른 AWS 서비스에 접근할 때 쓸 IAM 역할
    auto_create_execution_role=False,     # 역할을 자동 생성하지 않고 위 ARN 을 그대로 사용
    auto_create_ecr=True,                 # 컨테이너 이미지를 담을 ECR 저장소는 자동 생성
    requirements_file="requirements.txt", # 컨테이너 안에 설치할 의존성 목록
    region=region,
    agent_name=agent_name,
    authorizer_configuration=auth_config, # 위에서 만든 JWT 인증 설정 연결
)
print("Configuration completed ✓")
# 참고: linux/amd64 vs linux/arm64 플랫폼 불일치 경고가 떠도 무시해도 된다.
# AgentCore Runtime 은 ARM64(Graviton)를 쓰며, AWS 가 배포 시 올바른 아키텍처로 이미지를 다시 빌드한다.
# Note: You may see a platform mismatch warning (linux/amd64 vs linux/arm64).
# This is expected and can be safely ignored. AgentCore Runtime uses ARM64/Graviton
# processors, and AWS will automatically rebuild the image for the correct architecture
# during deployment. All dependencies in this lab are compatible with ARM64.

**참고:** 구성 중에 플랫폼 불일치 경고가 표시될 수 있습니다.

```
⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
```

**이 경고는 무시해도 됩니다.** AgentCore Runtime은 성능 및 비용 효율성 향상을 위해 AWS Graviton (ARM64) 프로세서를 사용합니다. AWS는 배포 프로세스 중에 올바른 ARM64 아키텍처에 맞게 Docker 이미지를 자동으로 재구축합니다. 이 실습에서 사용되는 모든 Python 종속성은 ARM64와 완벽하게 호환됩니다.

### 태스크 3.3.2: AgentCore Runtime에 에이전트 실행

이제 Dockerfile이 생겼으니 에이전트를 AgentCore Runtime으로 실행해 보겠습니다. 그러면 Amazon ECR 리포지토리와 AgentCore Runtime이 생성됩니다.

In [ ]:
# [배포 실행] 컨테이너 이미지를 빌드\uff65푸시하고 AgentCore Runtime 을 띄운다(수 분 소요).
print("Launching Agent server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch(
    env_vars={
        # OTEL_...: 관찰성 추적에서 헬스체크\uff65호출 경로를 제외해 노이즈를 줄임
        "OTEL_PYTHON_EXCLUDED_URLS": "/ping,/invocations",
        # 배포된 에이전트(main.py)가 런타임에 읽을 메모리 리소스 ID를 환경 변수로 전달.
        # memory_id 가 없으면 빈 문자열 -> main.py 는 메모리 없이 동작(점진적 성능 저하).
        "AGENTCORE_MEMORY_ID": memory_id if memory_id else ""
    }
)
print("Launch completed ✓")
# launch_result 에 배포 결과가 담긴다. 아래 호출 단계에서 agent_arn 을 쓴다.
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

#### 방금 무슨 일이 일어났을까요?

이 `agentcore_runtime.launch()` 명령은 몇 가지 중요한 작업을 수행했습니다.

1. **Amazon ECR 리포지토리 생성**: 에이전트의 Docker 이미지를 저장하기 위한 프라이빗 컨테이너 레지스트리
2. **Docker 이미지 구축**: 에이전트 코드, 종속성 및 구성을 컨테이너로 패키징
3. **ECR로 푸시됨**: 컨테이너 이미지를 ECR 저장소에 업로드
4. **AgentCore Runtime 생성**: 에이전트를 실행할 서버리스 런타임 환경을 배포
5. **구성된 환경 변수**: 메모리 통합 및 관찰성 설정

AWS 서비스를 쿼리하여 무엇이 생성되었는지 확인해 보겠습니다.

In [ ]:
import boto3
import json

# Initialize AWS clients
ecr_client = boto3.client('ecr', region_name=region)
agentcore_client = boto3.client('bedrock-agentcore-control', region_name=region)

print("=" * 80)
print("DEPLOYED RESOURCES VERIFICATION")
print("=" * 80)

# 1. Verify ECR Repository
# [검증 1] 배포로 생성된 ECR 저장소(컨테이너 이미지 보관소)를 확인한다.
print("\n📦 Amazon ECR Repository:")
print("-" * 80)
try:
    # Extract repository name from agent ARN or use a pattern
    # AgentCore typically creates repos with a specific naming pattern
    repos = ecr_client.describe_repositories(maxResults=10)
    
    # Find the most recently created repository (likely our agent)
    if repos['repositories']:
        # [정렬] 저장소를 생성 시각(createdAt) 기준 내림차순 정렬 -> [0] 이 가장 최근(우리 것).
        latest_repo = sorted(repos['repositories'], 
                           key=lambda x: x['createdAt'], 
                           reverse=True)[0]
        
        print(f"Repository Name: {latest_repo['repositoryName']}")
        print(f"Repository URI: {latest_repo['repositoryUri']}")
        print(f"Created: {latest_repo['createdAt']}")
        
        # Get image details
        images = ecr_client.describe_images(
            repositoryName=latest_repo['repositoryName'],
            maxResults=5
        )
        print(f"\nContainer Images: {len(images['imageDetails'])} image(s)")
        # [반복문] 최근 이미지 최대 3개를 순회하며 태그･크기･푸시시각을 출력.
        for img in images['imageDetails'][:3]:  # Show up to 3 most recent
            tags = img.get('imageTags', ['<untagged>'])
            # [단위 변환] 바이트 -> MB (1024*1024 로 나눔).
            size_mb = img['imageSizeInBytes'] / (1024 * 1024)
            print(f"  • Tag: {tags[0]}, Size: {size_mb:.1f} MB, Pushed: {img['imagePushedAt']}")
    else:
        print("No ECR repositories found")
except Exception as e:
    print(f"Error retrieving ECR info: {e}")

# 2. Verify AgentCore Runtime
# [검증 2] 배포된 Runtime 을 목록에서 찾아 상태･버전･ARN 을 확인한다.
print("\n🤖 AgentCore Runtime:")
print("-" * 80)
try:
    # List all runtimes and find ours
    runtime_response = agentcore_client.list_agent_runtimes(maxResults=10)
    
    # Find our runtime by matching the agent_id from launch_result
    our_runtime = None
    # [반복문] 전체 Runtime 목록을 순회하며 우리 agent_id 와 일치하는 것을 찾는다.
    for runtime in runtime_response.get('agentRuntimes', []):
        if runtime['agentRuntimeId'] == launch_result.agent_id:
            our_runtime = runtime
            break
    
    if our_runtime:
        print(f"Runtime Name: {our_runtime.get('agentRuntimeName', 'N/A')}")
        print(f"Runtime ARN: {our_runtime['agentRuntimeArn']}")
        print(f"Runtime ID: {our_runtime['agentRuntimeId']}")
        print(f"Status: {our_runtime.get('status', 'N/A')}")
        print(f"Version: {our_runtime.get('agentRuntimeVersion', 'N/A')}")
        print(f"Last Updated: {our_runtime.get('lastUpdatedAt', 'N/A')}")
        if our_runtime.get('description'):
            print(f"Description: {our_runtime['description']}")
    else:
        print(f"Runtime not found in list. Showing launch details:")
        print(f"  • Agent ARN: {launch_result.agent_arn}")
        print(f"  • Agent ID: {launch_result.agent_id}")
        print(f"\nNote: Runtime may still be initializing. Total runtimes found: {len(runtime_response.get('agentRuntimes', []))}")
            
except Exception as e:
    print(f"Error retrieving AgentCore Runtime info: {e}")
    print(f"\nShowing runtime details from launch:")
    print(f"  • Agent ARN: {launch_result.agent_arn}")
    print(f"  • Agent ID: {launch_result.agent_id}")

print("\n" + "=" * 80)
print("✅ Verification Complete")
print("=" * 80)
print("\nYour agent is now deployed and ready to be invoked!")

### 태스크 3.4: AgentCore Runtime 호출

마지막으로 페이로드를 사용하여 AgentCore Runtime을 간접적으로 호출할 수 있습니다.

In [ ]:
# [인증] Cognito 로 로그인해 베어러 토큰(JWT)을 받는다.
# 이 토큰을 HTTP 헤더 Authorization: Bearer <토큰> 에 넣어 배포된 에이전트를 호출한다.
bearer_token = reauthenticate_user(
    client_id=cognito_config["client_id"],
    secret_name=cognito_config["secret_name"]
)

In [ ]:
def invoke_endpoint(
    agent_arn: str,
    payload,
    session_id: str,
    bearer_token: Optional[str],
    region: str = None,
    endpoint_name: str = "DEFAULT",
) -> Any:
    """Invoke agent endpoint using HTTP request with bearer token."""
    if region is None:
        region = boto3.Session().region_name
    # [URL 조립] ARN 에는 :/ 등 특수문자가 있어 URL 에 넣으려면 인코딩이 필요하다.
    # safe="" -> 슬래시까지 전부 %XX 로 인코딩(경로 세그먼트로 안전하게 삽입).
    escaped_arn = urllib.parse.quote(agent_arn, safe="")
    url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_arn}/invocations"
    # [헤더] Bearer 토큰으로 인증하고, 세션 ID 를 전용 헤더로 전달한다.
    headers = {
        "Authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }

    try:
        # [입력 파싱] payload 가 문자열이면 JSON 으로 파싱, 이미 dict 면 그대로 사용.
        body = json.loads(payload) if isinstance(payload, str) else payload
    except json.JSONDecodeError:
        body = {"payload": payload}

    try:
        # [API 호출] 배포된 에이전트 엔드포인트로 POST. stream=True 로 응답을 조각조각 받는다.
        response = requests.post(
            url,
            params={"qualifier": endpoint_name},
            headers=headers,
            json=body,
            timeout=100,
            stream=True,
        )
        # [SSE 파싱] 서버-전송-이벤트(text/event-stream) 를 한 줄씩 읽어 텍스트만 뽑아낸다.
        # last_data: 직전 줄이 'data:' 였는지 기억해, 이어지는 줄 앞에 줄바꿈을 넣을지 판단.
        last_data = False
        # [반복문] 응답 스트림을 줄 단위로 순회(chunk_size=1 -> 도착하는 대로 즉시).
        for line in response.iter_lines(chunk_size=1):
            if line:
                line = line.decode("utf-8")
                # 'data: ' 로 시작하는 줄이 실제 응답 조각. line[6:] 로 'data: ' 접두어(6글자)를 잘라낸다.
                if line.startswith("data: "):
                    last_data = True
                    line = line[6:]
                    line = line.replace('"', "")
                    yield line
                elif line:
                    line = line.replace('"', "")
                    if last_data:
                        yield "\n" + line
                    last_data = False

    # [에러 처리] 네트워크･HTTP 오류를 잡아 로그를 남기고 다시 raise(호출 측에 전달).
    except requests.exceptions.RequestException as e:
        print("Failed to invoke agent endpoint: %s", str(e))
        raise

In [ ]:
# [최종 호출] 배포된 에이전트를 실제로 호출하고, 스트리밍 응답을 받아 출력한다.
# payload 에 prompt 와 actor_id 를 넣는다(actor_id 는 AgentCore Memory 의 사용자 구분 축).
for chunk in invoke_endpoint(
    agent_arn=launch_result.agent_arn,      # 배포 단계에서 받은 에이전트 ARN
    payload={
        "prompt": "I make $6000/month and want to start investing $500/month. Help me create a budget and suggest an investment portfolio.",
        "actor_id": "test_user_123"
    },
    session_id=str(uuid.uuid4()),           # 매 대화마다 고유 세션 ID(충돌 방지)
    bearer_token=bearer_token,
):
    # invoke_endpoint 는 제너레이터 -> 응답 조각(chunk)을 하나씩 받아 이어 붙여 출력.
    # "\\n" 이스케이프를 실제 줄바꿈으로 바꿔 보기 좋게 표시.
    print(chunk.replace("\\n", "\n"), end="")

**태스크 완료: **메모리 통합 기능이 있는 Amazon Bedrock AgentCore를 사용하여 다중 에이전트 시스템을 프로덕션에 성공적으로 배포했습니다. 재무 에이전트는 이제 다음과 같은 완전관리형 엔터프라이즈급 환경에서 실행됩니다.

- **AgentCore Runtime**: 자동 크기 조정, 관찰성 및 보안 기능 내장
- **AgentCore Memory**: 시맨틱, 사용자 선호도 및 요약 전략을 사용한 대화 전반의 컨텍스트 보존
- **프로덕션에 바로 사용할 수 있는 아키텍처**: 가드레일, 인증 및 오류 처리

메모리 통합을 통해 에이전트는 사용자 재무 정보, 선호도 및 대화 기록을 기억하고 세션 전반에 걸쳐 개인화된 조언을 제공할 수 있습니다. AWS에서 에이전틱 AI 애플리케이션의 개발부터 프로덕션 배포까지 전체 과정을 완료했습니다.

### 다음 단계

이 노트북을 완료했습니다. 실습의 다음 부분으로 넘어가려면 실습 지침으로 돌아가서 **태스크 4**를 계속하십시오.